### Local Settings ---- for installation on local computer ONLY

#### 1. Uv installation (local only, no need to redo if already done)


        https://docs.astral.sh/uv/getting-started/installation/


        `curl -LsSf https://astral.sh/uv/install.sh | sh`

        Python version 3.12 installation (highly recommended)
        `uv python install 3.12`


#### 3. Python env creation (local only)

        ```
        mkdir Projet_bias_mitigation
        cd Projet_bias_mitigation
        uv python pin 3.12
        uv init
        uv venv
        uv add numpy\
                torchvision\
                torch\
                torchmetrics\
                tensorboard\
                pillow\
                lightning\
                matplotlib\
                scikit_learn\
                ipykernel
        uv add pandas==2.2.2
        uv add setuptools==81.0
        ```

#### 4. In your folder "Projet_bias_mitigation"

        a. Copy your dataset and unzip it
                ```unzip YOUR_NAME.zip -d ./DATA/```
        b. Copy the "train_classifieur.py" in your folder

- - -
# Projet Final

> Arthur LE GAL, Leonor MARTIN-NOURRY, Carlos CLEMENT

> LDDIM3

> Faculte des sciences d'Orsay

# Plan du projet:

> Introduction

> Préparation et analyse des données

> Application des méthodes de pre processing et étude de métriques de fairness

> Application des méthodes de post processing et étude de métriques de fairness

> Conclusion


![Notation Projet](./notation.png)

- - -
# Introduction

In [29]:
#Utile uniquement sur colab, SINON ne pas lancer la cellule
from google.colab import drive
drive.mount('/content/drive')

import os
base = "/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness"
for f in os.listdir(base):
    print(f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train_classifieur.py
DATA
__pycache__
train_classifieur_local_Carlos.ipynb
expe_log


In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


#dataset = pd.read_csv("DATA/metadata.csv") #En local
dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata.csv") #Sur Colab
print(dataset.shape)
dataset.head()

(5581, 15)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,train_valid,label,WEIGHTS
0,00000008_000.png,Cardiomegaly,0,8,69,F,PA,2048,2500,0.171,0.171,NaN,train,malade,1
1,00000008_001.png,No Finding,1,8,70,F,PA,2048,2500,0.171,0.171,NaN,train,sain,1
2,00000008_002.png,Nodule,2,8,73,F,PA,2048,2500,0.168,0.168,NaN,train,malade,1
3,00000039_000.png,No Finding,0,39,76,M,PA,2500,2048,0.168,0.168,NaN,train,sain,1
4,00000039_001.png,No Finding,1,39,75,M,PA,2992,2991,0.143,0.143,NaN,train,sain,1


Comme au mi-projet, nous commençons par une prise en main du dataset.

Celui-ci contient 5581 lignes et 15 colonnes. On y retrouve les informations habituelles :

- l'identifiant de l'image radiographique,
- les diagnostics associés (ex : Mass|Nodule),
- le numéro de suivi du patient (0 = premier examen),
- l'identifiant du patient, son âge, son genre,
- la position de prise de vue (AP = antéro-postérieure, PA = postéro-antérieure),
- ainsi que les dimensions de l'image et la taille physique d'un pixel.

À cela s'ajoutent trois colonnes introduites pour le projet :

- train_valid (partition d'entraînement/validation),
- label (malade/sain)
- et WEIGHTS (poids d'échantillonnage).

Ce dataset nous permettra de chercher des corrélations,
des résultats notables et des biais grace a des informations médicales sur divers patients.

Pour simplifier l'analyse, nous ne retenons que 6 caractéristiques, en écartant l'identifiant image, les dimensions en pixels et la taille du pixel, qui n'apportent rien de pertinent ici.

- - -
# Preparation et analyse des données

> ## PREPARATION

In [31]:
dataset.dtypes #Pour voir avec quoi on travaille

,0
Image Index,object
Finding Labels,object
Follow-up #,int64
Patient ID,int64
Patient Age,int64
Patient Gender,object
View Position,object
OriginalImage[Width,int64
Height],int64
OriginalImagePixelSpacing[x,float64


On enleve Unnamed: 11, petite colonne d'erreur

In [32]:
dataset = dataset.drop(columns=["Unnamed: 11"])
print(dataset.columns.tolist())

['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'train_valid', 'label', 'WEIGHTS']


Au niveau de l'encodage des colonnes on procede de la meme facon que pour le mi-projet en ajoutant uniquement les colonnes supplementaires:

- train_valid : en one-hot encoding
- label : en one-hot encoding egalement
- WEIGHTS : pas de modification ici

In [33]:
selection = [
    "Finding Labels",
    "Follow-up #",
    "Patient ID",
    "Patient Age",
    "Patient Gender",
    "View Position",
    "train_valid",
    "label",
    "WEIGHTS"]

df = dataset[selection].copy()

df_test = df.copy() # Pour garder une copie avant modifications

for column in df.columns: #On verifie qu'il n'y ai pas de valeur NaN
    print(column, df[column].isnull().values.any())

labels = df["Finding Labels"].str.get_dummies(sep="|")
df = df.drop(columns=["Finding Labels"]).join(labels)

gender = pd.get_dummies(df["Patient Gender"], prefix="gender")
df = df.drop(columns=["Patient Gender"]).join(gender)

view = pd.get_dummies(df["View Position"], prefix="view")
df = df.drop(columns=["View Position"]).join(view)

train_valid = pd.get_dummies(df["train_valid"], prefix="train_valid")
df = df.drop(columns=["train_valid"]).join(train_valid)

label = pd.get_dummies(df["label"], prefix="label")
df = df.drop(columns=["label"]).join(label)

df.head()

Finding Labels False
Follow-up # False
Patient ID False
Patient Age False
Patient Gender False
View Position False
train_valid False
label False
WEIGHTS False


,Follow-up #,Patient ID,Patient Age,WEIGHTS,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,...,Pneumonia,Pneumothorax,gender_F,gender_M,view_AP,view_PA,train_valid_train,train_valid_valid,label_malade,label_sain
0,0,8,69,1,0,1,0,0,0,0,...,0,0,True,False,False,True,True,False,True,False
1,1,8,70,1,0,0,0,0,0,0,...,0,0,True,False,False,True,True,False,False,True
2,2,8,73,1,0,0,0,0,0,0,...,0,0,True,False,False,True,True,False,True,False
3,0,39,76,1,0,0,0,0,0,0,...,0,0,False,True,False,True,True,False,False,True
4,1,39,75,1,0,0,0,0,0,0,...,0,0,False,True,False,True,True,False,False,True


De la meme facon que lors du mi-projet on gere les ages car c'est la seule colonne avec des valeurs abherrantes.

In [34]:
print(f"Lignes avant : {len(df)}")
df = df[df["Patient Age"] <= 100]
print(f"Lignes après : {len(df)}")

Lignes avant : 5581
Lignes après : 5580


> ## ANALYSE

On va proceder comme durant le mi-projet en 3 etapes: structure du dataset, analyse statistique globale et analyse par groupe mais de facon plus breve

- ## Structure du dataset

On corrige le fait que nos maladies soit numeriques et non categorielles, sinon le dataset est correcte

In [35]:
numerical_features = list(df.select_dtypes(include=np.number).columns)

label_cols = ['Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion',
              'Emphysema','Fibrosis','Hernia','Infiltration','Mass','No Finding',
              'Nodule','Pleural_Thickening','Pneumonia','Pneumothorax']
numerical_features = [c for c in df.select_dtypes(include=np.number).columns if c not in label_cols]
numerical_features

['Follow-up #', 'Patient ID', 'Patient Age', 'WEIGHTS']

- ## Analyse Univariee

In [36]:
#Code permettant une analyse globale complete
from plotly.subplots import make_subplots
import plotly.graph_objects as go


df["sick"] = (df["No Finding"] == 0).astype(int) # Nouvelle colonne "sick": malade ou non, on l'utilisera par la suite

df_exam = df.copy() #On copie le dataset originel pour eviter des erreurs ou modifications sans faire expres
df_patient = (df_exam.sort_values("Follow-up #").drop_duplicates(subset="Patient ID", keep="last").copy()) # On garde a present pour l'analyse la table avec chaque patient de maniere unique, en preservant le follow up le plus eleve

print("Table de base :", df_exam.shape)
print("Table des patients :", df_patient.shape)

sickness = label_cols #liste des maladies
views = ["view_AP", "view_PA"] #les differents types de vue
sexes = ["gender_F", "gender_M"] #les differents genres
labels = ["label_malade", "label_sain"] #les differents labels
splits = ["train_valid_train", "train_valid_valid"] #les differents splits

fig = make_subplots(rows=3,
                    cols=2,
                    subplot_titles=("Age Counts", "Sickness counts", "View distribution", "Sex distribution", "Label distribution", "Train/Valid distribution"),
                    specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "domain"}, {"type": "domain"}], [{"type": "domain"}, {"type": "domain"}]],
                    horizontal_spacing=0.15,
                    vertical_spacing=0.15)

# 1
fig.add_trace(go.Histogram(x=df_patient["Patient Age"]), row=1, col=1)

# 2
counts4 = df_patient[sickness].sum().sort_values(ascending=False).reset_index()
counts4.columns = ["Sickness", "Count"]
fig.add_trace(go.Bar(x=counts4["Sickness"], y=counts4["Count"]), row=1, col=2)

# 3
counts5 = df_patient[views].sum().sort_values(ascending=False).reset_index()
counts5.columns = ["Views", "Count"]
fig.add_trace(go.Pie(labels=counts5["Views"], values=counts5["Count"]), row=2, col=1)

# 4
counts6 = df_patient[sexes].sum().sort_values(ascending=False).reset_index()
counts6.columns = ["Sexes", "Count"]
fig.add_trace(go.Pie(labels=counts6["Sexes"], values=counts6["Count"]), row=2, col=2)

# 5
counts7 = df_patient[labels].sum().sort_values(ascending=False).reset_index()
counts7.columns = ["Label", "Count"]
fig.add_trace(go.Pie(labels=counts7["Label"], values=counts7["Count"]), row=3, col=1)

# 6
counts8 = df_patient[splits].sum().sort_values(ascending=False).reset_index()
counts8.columns = ["Split", "Count"]
fig.add_trace(go.Pie(labels=counts8["Split"], values=counts8["Count"]), row=3, col=2)


fig.update_layout(title="All at once",
                  width=1200,
                  height=1200,
                  showlegend=False)

fig.show()

Table de base : (5580, 28)
Table des patients : (1500, 28)


L'histogramme des âges montre une distribution allant de 1 à environ 90 ans, avec une concentration marquée entre 40 et 65 ans   
et deux pics visibles, l'un vers 23 ans et l'autre vers 50 ans environ, un phénomène déjà observé au mi-projet, probablement lié   
à des effets d'arrondi dans la saisie des âges.  

On note un déséquilibre : les enfants et les très jeunes adultes sont sous-représentés, tout comme les patients très âgés.   
Ce biais d'âge est à garder en tête pour la suite, un modèle entraîné sur ce dataset risque de moins bien performer sur ces   
tranches d'âge.    

Concernant les pathologies, "No Finding" domine largement avec près de 1000 occurrences sur la table patients, soit environ les   
2/3 des cas, suivi de loin par Infiltration (environ 200) puis Effusion et Atelectasis (environ 100 chacun). Le dataset est donc    
fortement déséquilibré en faveur des patients sains, ce qui constitue un biais important à corriger lors de l'entraînement.    

La répartition des positions de vue est très déséquilibrée : 82.7% des images sont en vue PA (postéro-antérieure) contre seulement    
17.3% en AP (antéro-postérieure). C'est un biais notable par rapport au mi-projet où la répartition était plus équilibrée, le   
modèle pourrait apprendre des caractéristiques liées à la position de vue plutôt qu'aux pathologies elles-mêmes.    

La répartition par genre montre une légère surreprésentation masculine : 54.6% d'hommes contre 45.4% de femmes, soit un écart d'environ 9%.  

Enfin, les labels confirment le déséquilibre malade/sain déjà identifié (66.1% sain, 33.9% malade), et le split train/valid est de 75%/25%, ce qui est une répartition classique et équilibrée.  

- ## Analyse Bivariee

Tout comme pour le mi-projet, de maniere plus concsie et sur un seul plot, pour cette partie de l'etude, on va utiliser la table df_test qui est la table de base, car ca sera tout d'abord plus simple et car en general l'encodage one-hot est plutot utile pour entrainer un modele mais ce n'est pas ce que l'on fait ici. Regardons donc la fairness, pour commencer affichons les histogrammes des relations entre variables sensibles.

In [37]:
# On enleve les ages abherants sur la table de test rapidement pour la lisibilite des graphes
df_test = df_test[(df_test["Patient Age"] > 0) & (df_test["Patient Age"] <= 110)]

# Comme precedemment on garde un table avec une ligne par patients pour certaines analyses
df_test_patient = (df_test.sort_values("Follow-up #").drop_duplicates(subset="Patient ID", keep="last").copy())

print("Table test examens taille:", df_test.shape)
print("Table test patients taille:", df_test_patient.shape)

# Faisons des "bins" pour les variables numeriques des ages et des suivis.

#Bins pour les ages au niveau patients
df_test_patient["age_group"] = pd.cut(
    df_test_patient["Patient Age"],
    bins=[0, 17, 30, 65, 110],
    labels=["<=17", "18-30", "31-65", "65>"],
    include_lowest=True
)

# Bins pour les suivis au niveau examens
df_test["follow_group"] = pd.cut(
    df_test["Follow-up #"],
    bins = [1, 3, 10, 20, 50, 100, 150, 182], #On a vu 183 au debut du notebook
    labels = ["<=1", "2-3", "4-10", "11-20", "21-50", "51-100", "100>"],
    include_lowest=True
)

df_test["age_group"] = pd.cut( #Pour avoir les groupes d'age aussi dans df_test
    df_test["Patient Age"],
    bins=[0, 17, 30, 65, 110],
    labels=["<=17", "18-30", "31-65", "65>"],
    include_lowest=True
)

Table test examens taille: (5580, 9)
Table test patients taille: (1500, 9)


Apres avoir creer des tranches pour les ages ainsi que pour le nombre de suivis on va passer a la fonction principale qui fera l'integralite du travail. C'est a la base le code utilise en TD mais a present fortement modifie pour pouvoir generaliser le travail et faire tout sur des subplots.

In [38]:
df_test["sick"] = (df_test["Finding Labels"] != "No Finding").astype(int)
df_test_patient["sick"] = (df_test_patient["Finding Labels"] != "No Finding").astype(int)

def Display_categorical_hist(data, cat_feature, target="sick"):
    fig = px.histogram(data, x=cat_feature, color=target)
    return fig  # return au lieu de show

def Display_categorical_hist_percent(data, cat_feature, target="sick"):
    df_summarized = data.groupby([target, cat_feature]).size().reset_index(name="count")

    df_summarized[f"percent of {cat_feature}"] = (
        df_summarized.groupby(cat_feature)["count"]
        .transform(lambda c: 100 * c / c.sum())
    )

    df_summarized[target] = df_summarized[target].astype(str)
    fig = px.bar(df_summarized,
                 x=cat_feature,
                 y=f"percent of {cat_feature}",
                 color=target)

    return fig  # return au lieu de show


# Utilisation avec subplots

#for trace in f1.data:
#    fig.add_trace(trace, row=1, col=1)

def add_traces(fig, f, row, col):
    for trace in f.data:
        fig.add_trace(trace, row=row, col=col)

# Sick comme target
fig1 = make_subplots(rows=1, cols=4, subplot_titles=("Genre", "Vue", "Age group", "Follow group"))
add_traces(fig1, Display_categorical_hist_percent(df_test_patient, "Patient Gender"), 1, 1)
add_traces(fig1, Display_categorical_hist_percent(df_test_patient, "View Position"), 1, 2)
add_traces(fig1, Display_categorical_hist_percent(df_test_patient, "age_group"), 1, 3)
add_traces(fig1, Display_categorical_hist_percent(df_test, "follow_group"), 1, 4)
fig1.update_layout(title="Target: Sick", width=1400, height=400)
fig1.show()

# Genre comme target
fig2 = make_subplots(rows=1, cols=4, subplot_titles=("Sick", "Vue", "Age group", "Follow group"))
add_traces(fig2, Display_categorical_hist_percent(df_test_patient, "sick", "Patient Gender"), 1, 1)
add_traces(fig2, Display_categorical_hist_percent(df_test_patient, "View Position", "Patient Gender"), 1, 2)
add_traces(fig2, Display_categorical_hist_percent(df_test_patient, "age_group", "Patient Gender"), 1, 3)
add_traces(fig2, Display_categorical_hist_percent(df_test, "follow_group", "Patient Gender"), 1, 4)
fig2.update_layout(title="Target: Genre", width=1400, height=400)
fig2.show()

# Vue comme target
fig3 = make_subplots(rows=1, cols=4, subplot_titles=("Genre", "Sick", "Age group", "Follow group"))
add_traces(fig3, Display_categorical_hist_percent(df_test_patient, "Patient Gender", "View Position"), 1, 1)
add_traces(fig3, Display_categorical_hist_percent(df_test_patient, "sick", "View Position"), 1, 2)
add_traces(fig3, Display_categorical_hist_percent(df_test_patient, "age_group", "View Position"), 1, 3)
add_traces(fig3, Display_categorical_hist_percent(df_test, "follow_group", "View Position"), 1, 4)
fig3.update_layout(title="Target: Vue", width=1400, height=400)
fig3.show()

# Age group comme target
fig4 = make_subplots(rows=1, cols=4, subplot_titles=("Genre", "Vue", "Sick", "Follow group"))
add_traces(fig4, Display_categorical_hist_percent(df_test_patient, "Patient Gender", "age_group"), 1, 1)
add_traces(fig4, Display_categorical_hist_percent(df_test_patient, "View Position", "age_group"), 1, 2)
add_traces(fig4, Display_categorical_hist_percent(df_test_patient, "sick", "age_group"), 1, 3)
add_traces(fig4, Display_categorical_hist_percent(df_test, "follow_group", "age_group"), 1, 4)
fig4.update_layout(title="Target: Age group", width=1400, height=400)
fig4.show()

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to re

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to re

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to re

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_663/697352729.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to reta

### **Target: Sick**  
Le taux de malades est quasi identique entre hommes et femmes (environ 34%), confirmant l'analyse univariée,  
pas de biais de genre sur la maladie. En revanche, les patients en vue AP sont plus souvent malades que ceux en PA   
(environ 40% vs 33%), biais à surveiller. La tendance est claire sur l'âge : plus les patients sont âgés, plus le taux   
de malades augmente, passant de environ 28% chez les <=17 ans à environ 47% chez les 65>. Le numéro de suivi confirme   
aussi cette tendance, avec un taux de malades croissant jusqu'à environ 67% pour les 51-100 suivis.  

### **Target: Genre**  
La répartition H/F est stable quelle que soit la maladie ou la vue (environ 54% H, 46% F), sauf chez les 65> où les   
hommes représentent ~63%, c'est un biais d'âge croisé avec le genre. Les hommes accumulent aussi plus de suivis élevés   
que les femmes, ce qui combiné au biais précédent peut amener le modèle à associer "homme âgé" à "patient frequent".  

### **Target: Vue**  
La distribution AP/PA est très homogène selon le genre et la maladie (environ 18% AP, 82% PA partout). Le biais le plus   
notable est sur l'âge : les mineurs font près de 30% de vues AP contre 14-17% pour les autres tranches. Le modèle pourrait   
apprendre à associer vue AP avec jeunesse. Sur les suivis, la vue AP devient quasi exclusive à partir de 51 suivis,   
probablement liée à la mobilité réduite des patients en fin de parcours, biais fort à corriger.  

### **Target: Age group**  
La tranche 31-65 ans domine massivement pres de 70% dans toutes les variables, indépendamment du genre, de la vue ou de la   
maladie, c'est un déséquilibre de représentation important. À partir de 51 suivis, seule la tranche 31-65 est encore   
présente, ce qui signifie que le modèle n'aura quasiment aucun exemple de patients jeunes ou très âgés avec un suivi long.  

### **Biais à corriger en priorité pour l'entraînement :**  
Déséquilibre fort de la tranche d'âge 31-65, corrélation vue AP × âge jeune, corrélation suivi élevé × vue AP × maladie,   
et surreprésentation masculine chez les patients âgés.  

- - -
# Application des méthodes de pre-processing et étude de métriques de fairness

In [39]:
!pip install torchmetrics pytorch-lightning

In [40]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness')
#Les deux lignes precedentes servent a rediriger le path du systeme pour pour que colab trouve le fichier

from train_classifieur import train_classifier, pred_classifier
from datetime import datetime

In [41]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [42]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

Tesla T4
Memory Usage:
Allocated: 0.0 GB
Cached:    0.0 GB


In [43]:
# sauvegrade du dataset propre
chemin_csv_propre = "/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata_cleaned.csv" #EN colab
# chemin_csv_propre = "./DATA/metadata_cleaned.csv" #En local

df.to_csv(chemin_csv_propre, index=False)

print(f"Le dataset nettoyé sauvegardé physiquement sous : {chemin_csv_propre}")

Le dataset nettoyé sauvegardé physiquement sous : /content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata_cleaned.csv


- ## **Baseline**

###Entrainement du modele **baseline**

<mark> ATTENTION</mark> ne pas lancer la cellule suivante qui lance le modèle de prédiction sur le dataset.

In [44]:
# Liberation de la memoire GPU avant l'entrainement
import os
import torch

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

In [45]:
"""
%%time
# This cell show how to train a model

# You are going to train several time the models, with different weights

# The experiment folder name (logdir) suggested contains the timestamp
# you can change this to add some description in the name
# or add a file in the logdir folder to describe your intention and parameters
# Warning: For Colab users, the logdir needs to be on your drive (as in this example)
# It will allow you to keep your trained models, even if you get disconnected

# By default, to help you reproduce you experiments the csv file
# used for the training is copied in the logdir folder.

# This can take around 20-30min to run on PUIO computer
# And around 10 min on Colab with a GPU

nom_dossier = f"baseline_propre_{datetime.now().strftime('%Y_%m_%d_%H_%M_%S')}"
#logdir = f"./expe_log/{nom_dossier}" #En local
logdir = f"/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/expe_log/{nom_dossier}" #Sur colab


#Lancement de l'entraînement sur le fichier de propre
ckpt_path, ckpt_score = train_classifier(
    logdir=logdir,
    #datadir="./DATA/", #En local
    #csv="./DATA/metadata.csv", #En local
    datadir="/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/", #En colab
    csv="/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata.csv", #En colab
    weights_col="WEIGHTS",
    max_epochs=24
)

print(f"Entraînement terminé")
print(f"Meilleur modèle sauvegardé ici : {ckpt_path}")
print(f"Score (Balanced Accuracy) de validation : {ckpt_score}")
"""


'\n%%time\n# This cell show how to train a model\n\n# You are going to train several time the models, with different weights\n\n# The experiment folder name (logdir) suggested contains the timestamp\n# you can change this to add some description in the name\n# or add a file in the logdir folder to describe your intention and parameters\n# Warning: For Colab users, the logdir needs to be on your drive (as in this example)\n# It will allow you to keep your trained models, even if you get disconnected\n\n# By default, to help you reproduce you experiments the csv file\n# used for the training is copied in the logdir folder.\n\n# This can take around 20-30min to run on PUIO computer\n# And around 10 min on Colab with a GPU\n\nnom_dossier = f"baseline_propre_{datetime.now().strftime(\'%Y_%m_%d_%H_%M_%S\')}"\n#logdir = f"./expe_log/{nom_dossier}" #En local\nlogdir = f"/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/expe_log/{nom_dossier}" #Sur colab\n\n\n#Lancement de l\'entraîn

Après avoir exécuter la cellule ci-dessus on a pu observer que l'entrainement du modèle s'est arrêté après 23 epochs et un temps de 13min sur colab.

La cellule suivante lance TensorBoard, c'est un outil de visualisation qui affiche les courbes d'entraînement en temps réel (loss, accuracy, matrices de confusion) pendant ou après l'entraînement, c'est utile pour surveiller que le modèle apprend bien.

In [46]:
%load_ext tensorboard
%tensorboard --logdir=$logdir

ERROR: Failed to launch TensorBoard (exited with 2).
Contents of stderr:
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:292: SyntaxWarning: invalid escape sequence '\s'
  "[`\000-\040\177-\240\s]+",
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:339: SyntaxWarning: invalid escape sequence '\s'
  style = re.compile('url\s*\(\s*[^\s)]+?\s*\)\s*').sub(' ', style)
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:354: SyntaxWarning: invalid escape sequence '\s'
  if not re.match("^\s*([-\w]+\s*:[^:;]*(;\s*|$))*$", style):
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:358: SyntaxWarning: invalid escape sequence '\w'
  for prop, value in re.findall('([-\w]+)\s*:\s*([^:;]*)', style):
2026-03-30 17:23:16.744284: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one ha

On observe sur TensorBoard les courbes de validation du modèle baseline.  
La **balanced accuracy** plafonne autour de 0.62, avec de fortes oscillations, ce qui veut dire que le modèle apprend mais reste instable.

La **val_loss** descend jusqu'au step 80 environ puis remonte, signe clair d'overfitting i.e le modèle commence à trop apprendre les données d'entraînement et perd en généralisation.

Ces des résultats attendus pour un baseline entraîné sans aucune correction de biais i.e ou tous les poids sont à 1. Ces performances pas geniales justifient l'application des méthodes de pre-processing dans la suite du travail...

###Generation des predictions et de l'audit du modele **baseline**

In [47]:
%%time
# This cell show how to perform predictions with a train model
# the ckpt_path outputed in the previous cell refer to the best model obtained
# during the training, you can replace this by any .ckpt file obtained

#En local
"""
pred_classifier(
    datadir=f"./DATA/",
    csv_in=f"./DATA/metadata.csv",
    csv_out=f"{logdir}/preds.csv",
    ckpt_path = ckpt_path
)
"""

#En colab
pred_classifier(
    datadir="/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/",
    csv_in="/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata.csv",
    csv_out="/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/expe_log/preds_baseline.csv",
    ckpt_path=ckpt_path
)

NameError: name 'ckpt_path' is not defined

In [48]:
chemin_preds = "/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/expe_log/preds_baseline.csv" # ATTENTION: il faut remplacer par le bon chemin vers le fichier de predictions
df_preds = pd.read_csv(chemin_preds)

# Verification que pred_classifier a bien genere les colonnes preds, labels, preds_logit0, preds_logit1 etc... avant de faire l'audit
print("Colonnes disponibles :", df_preds.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/expe_log/preds_baseline.csv'

In [ ]:
# VARIABLES A ADAPTER SELON LE CSV
colonne_prediction = 'label'       # A Changer par le nom de la colonne de prediction du fichier
attribut_protege = 'View Position' # 1 si PA, 0 si AP
#---

def audit_modele(df, attr_protege, col_pred, priv_val='PA', unpriv_val='AP'):

    groupe_priv = df[df[attr_protege] == priv_val]     #groupe privilegie
    groupe_unpriv = df[df[attr_protege] == unpriv_val] #groupe non privilege

    #On calcule le taux de predictions positives ("etre malade") pour les deux groupes
    prob_pred_priv = (groupe_priv[col_pred] == 'malade').mean()
    prob_pred_unpriv = (groupe_unpriv[col_pred] == 'malade').mean()

    spd = prob_pred_unpriv - prob_pred_priv                                    # Statistical Parity Difference
    dir_ratio = prob_pred_unpriv / prob_pred_priv if prob_pred_priv > 0 else 0 # Disparate Impact Ratio

    print(f"=== Audit Baseline ===")
    print(f"'Malade' prédit pour {prob_pred_priv*100:.1f}% des radios {priv_val} et {prob_pred_unpriv*100:.1f}% des radios {unpriv_val}.")
    print(f"SPD : {spd:.4f}  (équité = 0.00)")
    print(f"DIR : {dir_ratio:.4f}  (équité = 1.00)")

audit_modele(df_preds, attribut_protege, colonne_prediction)

L'audit du modele sans pre-processing confirme que le réseau de neurones a appris et reproduit le biais d'acquisition présent dans les donnees d'entrainement. Le modèle a 11.13% (SPD = 0.1113) de chances en plus de predire qu'un patient est malade si la radio est prise en position AP plutôt qu'en PA.

Le Disparate Impact Ratio atteint 1.2594, ce qui depasse le seuil critique d'equite (fixe mathématiquement entre 0.8 et 1.25). Le modele Baseline est donc formellement considere comme injuste vis-à-vis de la position de la radiographie.

Pour rappel ce modele a ete entraine sur le dataset original sans aucune modification.

- ## **Pre-processing**

Comme lors du TD4 on va commencer par realiser l'opération inverse, soit creer un objet de la classe StandardDataset de AIF360 a partir du dataframe. Cela va nous permettre d'utiliser les méthodes deja implémentees dans AIF360 sur notre jeu de donnees.

In [49]:
!pip install aif360

In [70]:
from aif360.datasets import StandardDataset

df_aif = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata.csv")

# Creation d'un nouveau dataframe propre avec uniquement les colonnes necessaires
df_aif = pd.DataFrame({
    "label_num": (df_aif["label"] == "sain").astype(int),
    "ViewPosition": (df_aif["View Position"] == "PA").astype(int),
    "PatientGender": (df_aif["Patient Gender"] == "M").astype(int),
    "PatientAge": df_aif["Patient Age"],
    "FollowUp": df_aif["Follow-up #"],
    "WEIGHTS": df_aif["WEIGHTS"],
})

print(df_aif.head())

Dataset = StandardDataset(
    df=df_aif,
    label_name="label_num",
    favorable_classes=[1], #sain
    protected_attribute_names=["ViewPosition"],
    privileged_classes=[[1]], #PA
    instance_weights_name="WEIGHTS",
)

   label_num  ViewPosition  PatientGender  PatientAge  FollowUp  WEIGHTS
0          0             1              0          69         0        1
1          1             1              0          70         1        1
2          0             1              0          73         2        1
3          1             1              1          76         0        1
4          1             1              1          75         1        1


On vérifie dessous que le `StandardDataset` AIF360 represente bien les memes donnees que notre CSV original, en comparant le Disparate Impact et le Base Rate avec les metriques calculees precedemment.  
Si les chiffres sont coherents, on pourra passer au Reweighing.

In [71]:
from aif360.metrics import BinaryLabelDatasetMetric

print(
    BinaryLabelDatasetMetric(
        Dataset,
        unprivileged_groups=[{"ViewPosition": 0}],
        privileged_groups=[{"ViewPosition": 1}],
    ).disparate_impact(),

    BinaryLabelDatasetMetric(
        Dataset,
        unprivileged_groups=[{"ViewPosition": 0}],
        privileged_groups=[{"ViewPosition": 1}],
    ).base_rate(),
)

0.8050826014833847 0.5237412650062713


In [72]:
!pip install 'aif360[OptimalTransport]'

In [75]:
from aif360.sklearn.metrics import disparate_impact_ratio, base_rate

dir = disparate_impact_ratio(
    y_true=df_aif["label_num"], prot_attr=df_aif["ViewPosition"], pos_label=1, sample_weight=df_aif["WEIGHTS"] #pos_label = sain
)
br = base_rate(y_true=df_aif["label_num"], pos_label=1, sample_weight=df_aif["WEIGHTS"]) #pos_label = sain
dir, br

(0.8050826014833847, np.float64(0.5237412650062713))

> # METHODE 1

Le Disparate Impact et le Base Rate correspondent on passe donc a la reponderation:

In [77]:
unprivileged_groups = [{"ViewPosition": 0}]  # AP
privileged_groups = [{"ViewPosition": 1}]    # PA

# Application du Reweighing
from aif360.algorithms.preprocessing import Reweighing

RW = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

RW.fit(Dataset)

Dataset_transf = RW.transform(Dataset)

nouveaux_poids = Dataset_transf.instance_weights # Extraction des nouveaux poids

print("Nouveaux poids ->")
print("Min:", nouveaux_poids.min(), "| Max:", nouveaux_poids.max())
print("Moyenne:", nouveaux_poids.mean())

#Mise a jour de la colonne WEIGHTS dans le CSV original
df_original = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata.csv")
df_original["WEIGHTS"] = nouveaux_poids

# On sauvegarde
chemin_reweighed = "/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata_reweighed.csv"
df_original.to_csv(chemin_reweighed, index=False) #index = False evite les unnamed

Nouveaux poids ->
Min: 0.8813951725802626 | Max: 1.1394260792921362
Moyenne: 1.0


Comme precedemment on passe maintenant a l'entrainement du nouveau modele apres ajustement des poids.

- ## **Modele Repondere**

In [ ]:
"""
%%time
# This cell show how to train a model

# You are going to train several time the models, with different weights

# The experiment folder name (logdir) suggested contains the timestamp
# you can change this to add some description in the name
# or add a file in the logdir folder to describe your intention and parameters
# Warning: For Colab users, the logdir needs to be on your drive (as in this example)
# It will allow you to keep your trained models, even if you get disconnected

# By default, to help you reproduce you experiments the csv file
# used for the training is copied in the logdir folder.

# This can take around 20-30min to run on PUIO computer
# And around 10 min on Colab with a GPU

nom_dossier = f"baseline_propre_{datetime.now().strftime('%Y_%m_%d_%H_%M_%S')}"
#logdir = f"./expe_log/{nom_dossier}" #En local
logdir = f"/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/expe_log/{nom_dossier}" #Sur colab


#Lancement de l'entraînement sur le fichier de propre
ckpt_path, ckpt_score = train_classifier(
    logdir=logdir,
    #datadir="./DATA/", #En local
    #csv="./DATA/metadata.csv", #En local
    datadir="/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/", #En colab
    csv="/content/drive/MyDrive/Colab Notebooks/Projet_Final_Fairness/DATA/metadata.csv", #En colab
    weights_col="WEIGHTS",
    max_epochs=24
)

print(f"Entraînement terminé")
print(f"Meilleur modèle sauvegardé ici : {ckpt_path}")
print(f"Score (Balanced Accuracy) de validation : {ckpt_score}")
"""


###Generation des predictions et de l'audit du modele **Repondere**

> # Methode 2

- - -
# Application des méthodes de post processing et étude de métriques de fairness

- - -
# Analyse, compréhension de l'étude

- - -
# Conclusion